# 👔 Virtual Try-On — Colab (GPU grátis)

Corre o modelo **CatVTON** (treinado para try-on) na GPU do Colab e lança a app Gradio do projeto (`src/app.py`) com um **link público** para a demo.

**Antes de começar:** `Runtime > Change runtime type > T4 GPU`.

Fluxo: verificar GPU → clonar repos → instalar → carregar dados → teste rápido → lançar app.

## 1. Confirmar GPU

In [ ]:
!nvidia-smi

## 2. Clonar o repositório do projeto + o código do CatVTON

In [ ]:
import os, sys

# Repositório do projeto (contém src/ e a app).
if not os.path.exists('/content/repo'):
    !git clone https://github.com/goncalvesdaniel99/2026-ei-aoopii-a20776 /content/repo

# Código do CatVTON (a pipeline de try-on). Os pesos descarregam-se depois, do HF Hub.
if not os.path.exists('/content/CatVTON'):
    !git clone https://github.com/Zheng-Chong/CatVTON /content/CatVTON

# Tornar ambos importáveis.
for p in ('/content/CatVTON', '/content/repo/src'):
    if p not in sys.path:
        sys.path.insert(0, p)

print('OK')

## 3. Instalar dependências

Instalamos as versões **fixas** que a *pipeline* do CatVTON exige (`diffusers==0.29.2`,
`transformers==4.27.3`, …). As versões mais recentes do Colab quebram a API do CatVTON.

> **Importante:** depois desta célula, faz **Runtime → Restart session** e volta a correr a
> célula 2 (re-adiciona os caminhos ao `sys.path`; não volta a clonar).

Como usamos a **nossa** máscara (extraída do dataset), evitamos o `AutoMasker` do CatVTON
e, com ele, o `detectron2`/`densepose` — que são um pesadelo de instalar.

In [ ]:
!pip install -q diffusers==0.29.2 transformers==4.27.3 accelerate==0.31.0     huggingface_hub==0.23.4 gradio==4.44.1 opencv-python-headless einops

## 4. Dados (VITON-HD)

O `data/` está no `.gitignore`, por isso **não** veio no clone. Escolhe **uma** opção:

- **A) Upload de um zip** com a pasta `test/` (subpastas `image/`, `agnostic/`, `cloth/`).
- **B) Google Drive**, se já lá tiveres os dados.

Em qualquer dos casos, o objetivo é ter os dados em `/content/repo/data/test`.

In [ ]:
# Upload do data.zip e colocacao em /content/repo/data/test (deteta o caminho automaticamente).
from google.colab import files
import os, glob, shutil

up = files.upload()                 # escolhe o teu data.zip
zip_name = next(iter(up))
!rm -rf /content/_dataz && mkdir -p /content/_dataz
!unzip -q "{zip_name}" -d /content/_dataz

# encontra a pasta 'test' onde quer que tenha ficado dentro do zip
test_src = next(p for p in glob.glob('/content/_dataz/**/test', recursive=True)
                if os.path.isdir(os.path.join(p, 'image')))
os.makedirs('/content/repo/data', exist_ok=True)
dst = '/content/repo/data/test'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.move(test_src, dst)

DATA_DIR = '/content/repo/data/test'
assert os.path.isdir(os.path.join(DATA_DIR, 'image')), 'Dados em falta em ' + DATA_DIR
print('Dados OK:', sorted(os.listdir(DATA_DIR)))

## 5. Teste rápido (validar o modelo antes da UI)

Vestimos a **pessoa 0** com uma **peça diferente** para confirmar que o try-on é fiel
(preserva o desenho/cor da peça e adapta-a ao corpo). Primeira execução demora mais
(descarrega os pesos).

In [ ]:
from tryon import DatasetLoader, VirtualTryOn

data = DatasetLoader(DATA_DIR)
engine = VirtualTryOn(backend='catvton')   # auto também escolheria catvton (há GPU)

person_id = data.get_available_models()[0]
cloth_id  = data.get_available_garments()[5]   # peça diferente da original
print('Pessoa:', person_id, '| Peça:', cloth_id)

result = engine.generate(
    data.load_person(person_id),
    data.load_cloth(cloth_id),
    data.load_mask(person_id),
)
result

## 6. Lançar a app Gradio (com link público)

Usa a **tua** `src/app.py`. O `share=True` (via `GRADIO_SHARE=1`) gera um URL público
que podes abrir em qualquer máquina para a demo. Deixa esta célula a correr durante a demo.

In [ ]:
%env TRYON_BACKEND=catvton
%env GRADIO_SHARE=1
%env PYTHONPATH=/content/CatVTON
!cd /content/repo && python src/app.py

---
### Resolução de problemas

- **`No module named 'model'`** → a célula 2 não correu, ou `/content/CatVTON` não está no `PYTHONPATH` (célula 6).
- **OOM (out of memory)** na T4 → reduz a resolução no `CatVTONBackend` (ex. `width=512, height=768`) ou os `steps`.
- **Sessão expira** → o Colab grátis corta GPU por inatividade/tempo; reabre e volta a correr as células.
- **Máscara má** (a peça aparece em sítio errado) → ajusta `dilate`/`blur` em `DatasetLoader.load_mask`.